# 기본 베이스 라인

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_view_delete\Membership_v2.csv"

# 사용 컬럼 설정부
use_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "is_repurchase",
]

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

# 데이터 로드부
df = pd.read_csv(file_path, usecols=use_cols).copy()

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[
    [
        "price",
        "max_screen",
        "is_promotion",
        "is_churn_prevented",
        "payment_device",
        "is_user_verified",
        "gender",
        "age",
    ]
].copy()

# 양성 클래스 정의부
# is_repurchase == 0 을 예측 목표로 두기 때문에 0이면 1, 1이면 0으로 변환
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "age",
]

categorical_features = [
    "payment_device",
    "gender",
]

# 전처리 파이프라인 구성부
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    result = {
        "model": model_name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [["precision", "recall", "f1_score", "roc_auc", "pr_auc"]]
    .round(4)
    .sort_values("f1_score", ascending=False)
)

print("양성 클래스 기준: is_repurchase == 0")
print(results_df)

In [ ]:
import warnings
import pandas as pd

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# 실행 경고 정리부
warnings.filterwarnings(
    "ignore",
    message=(
        "X does not have valid feature names, but "
        "LGBMClassifier was fitted with feature names"
    ),
)

# 출력 형식 정리부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.expand_frame_repr", False)

# 파일 경로 설정부
file_path = (
    r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing"
    r"\260509_view_delete\Membership_v2.csv"
)

# 사용 컬럼 설정부
use_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "is_repurchase",
]

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

# 점수 계산 함수부
def evaluate_scores(y_true, y_pred, y_proba):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }

# 데이터 로드부
df = pd.read_csv(file_path, usecols=use_cols).copy()

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[
    [
        "price",
        "max_screen",
        "is_promotion",
        "is_churn_prevented",
        "payment_device",
        "is_user_verified",
        "gender",
        "age",
    ]
].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 정의부
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "age",
]

categorical_features = [
    "payment_device",
    "gender",
]

# 전처리 파이프라인 구성부
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 양성 클래스 가중치 계산부
positive_count = int(y_train.sum())
negative_count = int(len(y_train) - positive_count)
scale_pos_weight = negative_count / positive_count

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        class_weights=[1.0, scale_pos_weight],
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    # 학습 데이터 예측부
    y_train_pred = clf.predict(X_train)
    y_train_proba = clf.predict_proba(X_train)[:, 1]

    # 테스트 데이터 예측부
    y_test_pred = clf.predict(X_test)
    y_test_proba = clf.predict_proba(X_test)[:, 1]

    # 학습 데이터 점수 계산부
    train_scores = evaluate_scores(y_train, y_train_pred, y_train_proba)

    # 테스트 데이터 점수 계산부
    test_scores = evaluate_scores(y_test, y_test_pred, y_test_proba)

    # ROC AUC gap 계산부
    roc_auc_gap = train_scores["roc_auc"] - test_scores["roc_auc"]

    # 과적합 여부 판단부
    overfit_flag = "과적합 의심" if roc_auc_gap >= 0.03 else "과적합 아님"

    # 결과 저장부
    result = {
        "model": model_name,
        "precision": test_scores["precision"],
        "recall": test_scores["recall"],
        "f1": test_scores["f1"],
        "train_roc_auc": train_scores["roc_auc"],
        "test_roc_auc": test_scores["roc_auc"],
        "overfit": overfit_flag,
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [
        [
            "precision",
            "recall",
            "f1",
            "train_roc_auc",
            "test_roc_auc",
            "overfit",
        ]
    ]
    .round(
        {
            "precision": 4,
            "recall": 4,
            "f1": 4,
            "train_roc_auc": 4,
            "test_roc_auc": 4,
        }
    )
    .sort_values(["f1", "test_roc_auc"], ascending=False)
)

print("양성 클래스 기준: is_repurchase == 0")
print("precision, recall, f1 는 test 기준")
print("train_roc_auc, test_roc_auc 는 roc_auc_score 기준")
print("과적합 판단 기준: train_roc_auc - test_roc_auc >= 0.03")
print(results_df.to_string())


# 파생변수 추가

In [ ]:
import warnings
import pandas as pd

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# 실행 경고 정리부
warnings.filterwarnings(
    "ignore",
    message=(
        "X does not have valid feature names, but "
        "LGBMClassifier was fitted with feature names"
    ),
)

# 출력 형식 정리부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.expand_frame_repr", False)

# 파일 경로 설정부
file_path = (
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable"
    r"\260519_derived_membership_age_specific.csv"
)

# 원본 전체 컬럼 설정부
raw_source_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
]

# 원본 사용 컬럼 설정부
base_feature_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 타깃 컬럼 설정부
target_col = "is_repurchase"

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

# 점수 계산 함수부
def evaluate_scores(y_true, y_pred, y_proba):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }

# 입력 컬럼 추출 함수부
def get_feature_columns(df):
    missing_cols = [
        col for col in base_feature_cols + [target_col] if col not in df.columns
    ]
    if missing_cols:
        raise ValueError(f"필수 컬럼 누락: {missing_cols}")

    derived_feature_cols = [
        col for col in df.columns if col not in raw_source_cols
    ]

    feature_cols = base_feature_cols + derived_feature_cols

    return feature_cols

# 숫자형, 범주형 컬럼 분리 함수부
def split_feature_types(X):
    numeric_features = X.select_dtypes(include="number").columns.tolist()
    categorical_features = [
        col for col in X.columns if col not in numeric_features
    ]
    return numeric_features, categorical_features

# 데이터 로드부
df = pd.read_csv(file_path).copy()

# 사용 컬럼 추출부
feature_cols = get_feature_columns(df)

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df[target_col])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 정의부
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features, categorical_features = split_feature_types(X)

# 전처리 파이프라인 구성부
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 양성 클래스 가중치 계산부
positive_count = int(y_train.sum())
negative_count = int(len(y_train) - positive_count)
scale_pos_weight = negative_count / positive_count

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        class_weights=[1.0, scale_pos_weight],
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    # 학습 데이터 예측부
    y_train_pred = clf.predict(X_train)
    y_train_proba = clf.predict_proba(X_train)[:, 1]

    # 테스트 데이터 예측부
    y_test_pred = clf.predict(X_test)
    y_test_proba = clf.predict_proba(X_test)[:, 1]

    # 학습 데이터 점수 계산부
    train_scores = evaluate_scores(y_train, y_train_pred, y_train_proba)

    # 테스트 데이터 점수 계산부
    test_scores = evaluate_scores(y_test, y_test_pred, y_test_proba)

    # ROC AUC gap 계산부
    roc_auc_gap = train_scores["roc_auc"] - test_scores["roc_auc"]

    # 과적합 여부 판단부
    overfit_flag = "과적합 의심" if roc_auc_gap >= 0.03 else "과적합 아님"

    # 결과 저장부
    result = {
        "model": model_name,
        "precision": test_scores["precision"],
        "recall": test_scores["recall"],
        "f1": test_scores["f1"],
        "train_roc_auc": train_scores["roc_auc"],
        "test_roc_auc": test_scores["roc_auc"],
        "overfit": overfit_flag,
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [
        [
            "precision",
            "recall",
            "f1",
            "train_roc_auc",
            "test_roc_auc",
            "overfit",
        ]
    ]
    .round(
        {
            "precision": 4,
            "recall": 4,
            "f1": 4,
            "train_roc_auc": 4,
            "test_roc_auc": 4,
        }
    )
    .sort_values(["f1", "test_roc_auc"], ascending=False)
)

print(f"사용한 입력 컬럼 수: {len(feature_cols)}")
print("양성 클래스 기준: is_repurchase == 0")
print("precision, recall, f1 는 test 기준")
print("train_roc_auc, test_roc_auc 는 roc_auc_score 기준")
print("과적합 판단 기준: train_roc_auc - test_roc_auc >= 0.03")
print(results_df.to_string())
